### Tools

Models can request to call tools tha perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, abd/or argument definitions (often a JSON schema)
2. A function or coroutine to execute

In [13]:
import os
from langchain.chat_models import init_chat_model

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

model = init_chat_model("google_genai:gemini-flash-lite-latest")

response = model.invoke("Hello, how are you?")
response

AIMessage(content=[{'type': 'text', 'text': "Hello! I'm doing well, thank you for asking. How can I help you today?", 'extras': {'signature': 'El4KXAERTTIP9X8/i+KPsVvhkudV8eGm6rKnX494dqiH9EvkVvtPd92rZk+AT6R19Xj2gsqvjf77J7PeMvVqAaXH7LcBxSO58ZC/HoVOciEXcfOY1inJK2fL5fmZLPY8'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a06f22-5265-79c3-8274-27af5a38c7f7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 7, 'output_tokens': 20, 'total_tokens': 27, 'input_token_details': {'cache_read': 0}})

In [8]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """
    Get the current weather for a given location.
    """
    return f"The current weather in {location} is sunny."  

# Alternative way for 1-langchainintro to bind tools to the model
model_with_tools = model.bind_tools([get_weather])

response = model_with_tools.invoke("What is the weather in New York?")
print(response)

for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Arguments: {tool_call['args']}")

content=[] additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "New York"}'}, '__gemini_function_call_thought_signatures__': {'call_4450': 'El4KXAERTTIP3uHUfmuFkkuJzUc0mM8QsgrDHesf/QA8I6bR2klTng+gb/Se7uAtuCozC7ujN3hBJsDxZwYz7vzBIi18CXyVYnP7qR7Zcz7qJJviEJjWwTQP8U03uk6E'}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a06f0f-049e-7f40-8908-8431a8912f87-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'call_4450', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 55, 'output_tokens': 17, 'total_tokens': 72, 'input_token_details': {'cache_read': 0}}
Tool: get_weather
Arguments: {'location': 'New York'}


### Tool Execution Loop

In [ ]:
# Step 1: Model generates tool calls
messages = [
    {"role": "user", "content": "What is the weather in New York?"}
]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to the model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)


The current weather in New York is sunny.


[{'role': 'user', 'content': 'What is the weather in New York?'},
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "New York"}'}, '__gemini_function_call_thought_signatures__': {'call_8205': 'El4KXAERTTIPtgSc27bSib3DKuWUo27Pl4V+qmU//duFFB5WdGelmGdU8ge1Hi9uP5TK/gRcAfb/TWnnviTg8mVWAuZyQD5jBhs18R4Hn56GwHMIIRDfRVsvLKaBLlwi'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a06f1a-39af-7561-be72-582445b7f60f-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'call_8205', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 55, 'output_tokens': 17, 'total_tokens': 72, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content='The current weather in New York is sunny.', name='get_weather', tool_call_id='call_8205')]